In [5]:
import numpy as np
import pandas as pd

# Task 1

In [6]:
np.random.seed(42)
n = 400

bikes = pd.DataFrame({
    'listing_id': range(1, n + 1),
    'brand': np.random.choice(['Veloce', 'Rajhans', 'Sundar', 'Tarang'], n),
    'city': np.random.choice(['Ahmedabad', 'Surat', 'Vadodara', 'Rajkot'], n),
    'model_year': np.random.randint(2008, 2024, n),
    'km_driven': np.random.randint(2000, 90001, n).astype(float),
    'owner_count': np.random.randint(1, 4, n).astype(float),
    'price_inr': np.random.randint(25000, 140001, n).astype(float)
})

price_cutoff = bikes['price_inr'].quantile(0.95)
bikes.loc[bikes['price_inr'] > price_cutoff, 'price_inr'] = np.nan

km_blank = np.random.choice(bikes.index, int(0.08 * n), replace=False)
bikes.loc[km_blank, 'km_driven'] = np.nan

old_rows = bikes[bikes['model_year'] < 2013].index
owner_blank = np.random.choice(old_rows, len(old_rows) // 2, replace=False)
bikes.loc[owner_blank, 'owner_count'] = np.nan

bikes.to_csv('bike_listings_raw.csv', index=False)
print(bikes.shape)
print(bikes.isna().sum())

(400, 7)
listing_id      0
brand           0
city            0
model_year      0
km_driven      32
owner_count    67
price_inr      20
dtype: int64


# Task 2

In [7]:
raw = pd.read_csv('bike_listings_raw.csv')

missing_count = raw.isna().sum()
missing_report = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': (missing_count / len(raw) * 100).round(2)
})
missing_report['mechanism'] = missing_report.index.map({'km_driven': 'MCAR', 'owner_count': 'MAR', 'price_inr': 'MNAR'}).fillna('None')
print(missing_report)

complete_rows = raw.notnull().all(axis=1).sum()
print("Fully complete rows:", complete_rows)

old_rate = raw[raw['model_year'] < 2013]['owner_count'].isna().mean() * 100
new_rate = raw[raw['model_year'] >= 2013]['owner_count'].isna().mean() * 100
print("owner_count missing rate, model_year below 2013:", round(old_rate, 2), "%")
print("owner_count missing rate, model_year 2013 and later:", round(new_rate, 2), "%")

             missing_count  missing_pct mechanism
listing_id               0         0.00      None
brand                    0         0.00      None
city                     0         0.00      None
model_year               0         0.00      None
km_driven               32         8.00      MCAR
owner_count             67        16.75       MAR
price_inr               20         5.00      MNAR
Fully complete rows: 289
owner_count missing rate, model_year below 2013: 49.63 %
owner_count missing rate, model_year 2013 and later: 0.0 %


# Task 3

In [8]:
drop_any = raw.dropna(how='any')
drop_thresh = raw.dropna(thresh=6)
drop_cols = raw.dropna(axis=1, thresh=360)

comparison = pd.DataFrame({
    'rows': [raw.shape[0], drop_any.shape[0], drop_thresh.shape[0]],
    'columns': [raw.shape[1], drop_any.shape[1], drop_thresh.shape[1]],
    'mean_price_inr': [raw['price_inr'].mean(), drop_any['price_inr'].mean(), drop_thresh['price_inr'].mean()]
}, index=['raw', 'drop_any', 'drop_thresh'])

print(comparison)
print("drop_cols columns:", list(drop_cols.columns))

             rows  columns  mean_price_inr
raw           400        7    79096.457895
drop_any      289        7    80949.539792
drop_thresh   392        7    79090.135638
drop_cols columns: ['listing_id', 'brand', 'city', 'model_year', 'km_driven', 'price_inr']


# Task 4

In [9]:
bikes_filled = raw.copy()

bikes_filled['km_driven'] = bikes_filled['km_driven'].fillna(bikes_filled['km_driven'].median())
bikes_filled['owner_count'] = bikes_filled['owner_count'].fillna(bikes_filled['owner_count'].mode()[0])

bikes_filled = bikes_filled.sort_values('model_year')
bikes_filled['price_inr'] = bikes_filled['price_inr'].ffill()
bikes_filled['price_inr'] = bikes_filled['price_inr'].bfill()

print("Mean price, raw non-missing rows:", raw['price_inr'].mean())
print("Mean price, drop_any:", drop_any['price_inr'].mean())
print("Mean price, bikes_filled:", bikes_filled['price_inr'].mean())

print("Missing values left:", bikes_filled.isna().sum().sum())
bikes_filled.to_csv('bike_listings_s6.csv', index=False)

Mean price, raw non-missing rows: 79096.45789473684
Mean price, drop_any: 80949.53979238754
Mean price, bikes_filled: 78436.8725
Missing values left: 0
